In [78]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [35]:
loader = PyPDFLoader("langchain_info.pdf")
pdf = loader.load()

In [40]:
print(pdf)

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'langchain_info.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tools for han-\ndling chat models, integrating retr

In [41]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [43]:
chunks = splitter.split_documents(pdf)

In [44]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'langchain_info.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tools for han-\ndling chat models, integrating retr

In [47]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 864.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
vector_store.index_to_docstore_id

{0: '2733eac9-2b52-462e-9205-59e28baf64e1',
 1: 'a6ddaae6-17ab-4343-a14c-85bfd7aaf46d',
 2: 'c2bc2efa-6dc4-4fd3-8758-ec26dc20b4ba',
 3: 'ba1354b4-1bb9-4b08-a0e9-b4d853127810',
 4: 'b438aa3c-f40a-4dd3-aa9e-17bd91081ec1',
 5: '919a145d-28dd-477a-a264-a77e436d9176',
 6: 'aa619bfe-8d9a-4533-8fe9-0d8a83f4583c',
 7: '15657f95-8d1a-4f39-bc26-24c788f1c404',
 8: '881ef450-7812-4fa1-a769-4251423f3bc8',
 9: 'ca9fe8a1-e76c-4f20-8958-40b508b0d8be',
 10: 'd7167d8f-71d7-4d2c-b7fd-6de028298780',
 11: '3089e195-1dab-4acb-be88-a74b208b6c9f',
 12: 'a96398bc-5113-4e91-9654-d7edf9b17ab5',
 13: 'cc4d2e0e-8aaa-46a4-9d36-c5a017034b8e',
 14: '047f869b-bede-4d51-aa77-21965366fedb',
 15: '6a052b66-fe72-478a-b716-b6d4a73ebf7d',
 16: '2fdcebb2-c853-48ac-bb4b-547b2143d899',
 17: 'e155a87f-ed80-44dc-bd95-47e63a8ae8d5',
 18: '416cc346-7613-44db-a7ac-90f565ec31ea',
 19: 'a77e1b09-cde0-46b8-92a2-e7b0e136ea9c',
 20: '75b744db-48c2-4ff4-9956-fdd4b4c38d25',
 21: '754e14fc-9862-458b-ba10-607144d6f984',
 22: '0288162c-df9d-

In [61]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [62]:
retriever.invoke("what consist this pdf")

[Document(id='15657f95-8d1a-4f39-bc26-24c788f1c404', metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'langchain_info.pdf', 'total_pages': 14, 'page': 2, 'page_label': '3'}, page_content='LangChain 3\nneeds, providing a flexible foundation for building scalable, secure, and multi-\nfunctional applications. Figure 1 illustrates a fundamental LangChain pipeline.\nIn this architecture, diverse data sources—including documents, text, and im-\nages—are embedded and stored within a vector store. Upon receiving a user’s\nquery, the system retrieves the most relevant information from the vector store.\nThis retrieved context is then provided to the large language model (LLM),\nenhancing its 

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="use_your_apikey"
)

In [92]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided PDF context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [93]:
question = "what consist this pdf? "
retrieved_docs = retriever.invoke(question)

In [94]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'LangChain 3\nneeds, providing a flexible foundation for building scalable, secure, and multi-\nfunctional applications. Figure 1 illustrates a fundamental LangChain pipeline.\nIn this architecture, diverse data sources—including documents, text, and im-\nages—are embedded and stored within a vector store. Upon receiving a user’s\nquery, the system retrieves the most relevant information from the vector store.\nThis retrieved context is then provided to the large language model (LLM),\nenhancing its ability to generate accurate and factually grounded responses.\nFig. 1. LangChain pipeline architecture showcasing the retrieval-augmented genera-\ntion process. Documents in various formats (e.g., PDF, text, images) are preloaded\nand embedded into a vector store. When a user submits a query, the system retrieves\nthe top-k most relevant documents based on vector similarity. These documents are\ncombined with the query to provide contextual information to the language model\n\nLachaux, Tim

In [95]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [96]:
answer = llm.invoke(final_prompt)
print(answer.content)

This PDF consists of information about LangChain, a framework for building scalable and secure applications, including:

1. LangChain pipeline architecture
2. Vector store and retrieval-augmented generation process
3. Large language models (LLMs)
4. PromptTemplates for standardizing and formatting queries
5. Features such as background execution, support for long-running agents, burst handling, and task queues
6. LangGraph workflow, including defining state schema, nodes, and transitions between nodes.


In [97]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [98]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [100]:
parser = StrOutputParser()

In [101]:
main_chain = parallel_chain | prompt | llm | parser

In [102]:
main_chain.invoke('which topic covered in this video')

"The topics covered in this context appear to be related to LangGraph and LangChain, specifically their features, functionalities, and security models. Some of the topics mentioned include:\n\n1. Stateful execution model and persistent state\n2. Human-in-the-loop interactions\n3. Retrieval-Augmented Generation (RAG)\n4. Security model and areas for improvement\n5. LangChain's features, such as multi-turn interactions, structured output, and conversation memory."